In [362]:
#import libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [363]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATASET")
    print("=" * 60)

    df = pd.read_csv(file_path)
    print("Shape of the dataset:")
    print(df.shape)
    print("\ncheck for missing value")
    print(df.isnull().sum())
    print("\First five five row")
    print(df.head())
    print("\Descriptive statistics")
    print(df.describe())
    print("\Dataset Info:")
    print(df.info())
    print("\Condition Distrubtion")
    print(df['condition'].value_counts())
    print("\Transmission Distributional")
    print(df['transmission'].value_counts())

    return df

In [364]:
def preprocessing_data(df):
    print("\n" + "=" * 60)
    print("PREPROCESSING DATASET")
    print("=" * 60)

    df_preprocessed = df.copy()
    df_preprocessed = df.dropna()

    label_encoder = {}
    df_columns = ['make', 'model', 'transmission', 'condition']
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder[col] = le
        print(f"\n{col} encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label} : {i}")
    print("Processed data shape:", df_preprocessed.shape)  
    
    return df_preprocessed, label_encoder

In [365]:
def feature_data(df_preprocessed):
    print("\n" + "=" * 60)
    print("FEATURE DATA")
    print("=" * 60)

    feature_columns = ['year','make_encoded','model_encoded','transmission_encoded','condition_encoded']
    target = ['price']

    X = df_preprocessed[feature_columns]
    y = df_preprocessed[target]

    print("\nFeature Shape:", df_preprocessed[feature_columns].shape)
    print("\nTarget Shape:", df_preprocessed[target].shape)
    print("\nFeatures Column:", feature_columns)

    return X,y, feature_columns

In [366]:
def split_data(X,y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITTING DATA")
    print("=" * 60) 

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state = random_state
    )

    print("\Training set size:", X_train.shape[0])
    print("\Testing set size:", X_test.shape[0])
    print("\nTraining set price range {:,.2f} - {:,.2f}".format(
        float(y_train.min()), float(y_train.max())
    ))
    print("\nTesting set price range {:,.2f} - {:,.2f}".format(
        float(y_test.min()), float(y_test.max())
    ))

    return X_train, X_test, y_train, y_test

In [367]:
def scale_feature(X_train, X_test):
    print("\n" + "=" * 60)
    print("SCALE FEATURE DATA")
    print("=" * 60) 

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print('\Feature scaled successfully')
    print('\Training set scaled:', X_train_scaled.shape)
    print('\Testing set scaled:', X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, scaler

In [368]:
def train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)
    
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    print("\nModel train successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(feature_columns, model.coef_.ravel()):
        print(f" {feature} : {coef:.2f}")
    # print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")
    
    return model

In [393]:
def train_random_forest_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("TRAINING RANDOM FOREST MODEL")
    print("=" * 60)

    rm_model = RandomForestRegressor(
        n_estimators = 100,
        max_depth = 15,
        min_samples_split = 5,
        min_samples_leaf = 2,
        random_state = 42,
        n_jobs = 1
    )

    rm_model.fit(X_train_scaled, y_train)
    print("\nRandom forest model trained successfully")
    print("Model estimator:", rm_model.estimator)
    print("Model max depth:", rm_model.max_depth)

    feature_importance = sorted(zip(feature_columns, rm_model.feature_importances_),
                             key= lambda x:x[1], reverse = True)

    for feature, importance in feature_importance:
        print(f" {feature}: {float(importance):.4f}")

    return rm_model    

In [394]:
def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUATING MODEL")
    print("=" * 60)

    #make prediction
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)
    print("MODEL PERFORMANCE")
    print("=" * 60)
    print("\nTraining Set")
    print(f" R2 Score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 Score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")
    print("=" * 60)

    #cross-validation
    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print("\n Cross Validation(5 folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross Validation mean: {cv_score.mean():.4f}")
    print(f" Cross Validation std: {cv_score.std():.4f}")

    joblib.dump(test_rmse, "model_rmse.pkl")
    joblib.dump(test_mae, "model_mae.pkl")
    print("Error saved successfully")
    
    metrics = {
        "train_r2": train_r2,
        "test_r2" : test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score":  cv_score
    }
        
    return metrics

In [406]:
def evaluate_rm_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUATING RANDOM FOREST MODEL")
    print("=" * 60)

    #make prediction
    y_train_pred = rm_model.predict(X_train_scaled)
    y_test_pred = rm_model.predict(X_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)
    print("RANDOM FOREST MODEL PERFORMANCE")
    print("=" * 60)
    print("\nTraining Set")
    print(f" R2 Score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 Score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")
    print("=" * 60)

    #cross-validation
    cv_score = cross_val_score(rm_model, X_train_scaled, y_train, cv=5, scoring='r2')
    print("\n Cross Validation(5 folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross Validation mean: {cv_score.mean():.4f}")
    print(f" Cross Validation std: {cv_score.std():.4f}")

    joblib.dump(test_rmse, "model_rmse.pkl")
    joblib.dump(test_mae, "model_mae.pkl")
    print("Error saved successfully")
    
    metrics = {
        "train_r2": train_r2,
        "test_r2" : test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score":  cv_score
    }
        
    return metrics

In [407]:
def save_model_artifact(model, scaler, label_encoder, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "car_price_prediction.pkl") 
    print("Car price prediction saved successfully")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler features saved successfully")

    joblib.dump(label_encoder, "label_encoder.pkl")     
    print("Label encoder saved successfully")

    joblib.dump(feature_columns, "feature_columns.pkl")
    print("Feature columns saved successfully")

    print("\n" + "=" * 60)
    print("ALL MODEL ARTIFACT SAVED SUCCESSFULLY")
    print("=" * 60)
    

In [416]:
def save_rm_model_artifact(rm_model):
    print("\n" + "=" * 60)
    print("SAVING RANDOM FOREST MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_prediction.pkl") 
    print("Car price prediction saved successfully")

    print("\n" + "=" * 60)
    print("RANDOM FOREST MODEL ARTIFACT SAVED SUCCESSFULLY")
    print("=" * 60)
    

In [413]:
def predict_car_price(year, make, model_name, transmission, condition):
    model = joblib.load("car_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")
    rmse = joblib.load("model_rmse.pkl")

    try:
        make_encoded = label_encoder["make"].transform([make])[0]
        model_encoded = label_encoder["model"].transform([model_name])[0]
        transmission_encoded = label_encoder["transmission"].transform([transmission])[0]
        condition_encoded = label_encoder["condition"].transform([condition])[0]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
         "year": year,
         "make_encoded" : make_encoded,
         "model_encoded" : model_encoded,
         "transmission_encoded" : transmission_encoded,
         "condition_encoded" : condition_encoded 
     }   

    features = np.array([[features_dict[col] for col in features_dict]])
    
    feature_scale = scaler.transform(features)

    predicted_price = model.predict(feature_scale)[0]

    lower = predicted_price - rmse
    upper = predicted_price + rmse

    # return (lower.item(), predicted_price.item(), upper.item())
    return (lower.item(), upper.item())

In [427]:
def rm_model_predict_car_price(year, make, model_name, transmission, condition):
    rm_model = joblib.load("rm_model_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")
    rmse = joblib.load("model_rmse.pkl")

    try:
        make_encoded = label_encoder["make"].transform([make])[0]
        model_encoded = label_encoder["model"].transform([model_name])[0]
        transmission_encoded = label_encoder["transmission"].transform([transmission])[0]
        condition_encoded = label_encoder["condition"].transform([condition])[0]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
         "year": year,
         "make_encoded" : make_encoded,
         "model_encoded" : model_encoded,
         "transmission_encoded" : transmission_encoded,
         "condition_encoded" : condition_encoded 
     }   

    features = np.array([[features_dict[col] for col in features_dict]])
    
    feature_scale = scaler.transform(features)

    predicted_price = rm_model.predict(feature_scale)[0]

    lower = predicted_price - rmse
    upper = predicted_price + rmse

    return predicted_price.item()

In [428]:
def test_prediction():
    print("\n" + "=" * 60)
    print("TESTING PREDICTION")
    print("=" * 60)

    # #Example 1
    # price1 = predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    # print("\n 2015 Toyota Camry( Local used Automatic)")
    # print(f"  ₦{float(price1):,.2f}")
        
    # #Example 2
    # price2 = predict_car_price(2010, "Honda", "Accord", "Automatic", "Local Used")
    # print("\n 2010 Honda Accord ( Local used Automatic)")
    # print(f"  ₦{float(price2):,.2f}")

    # #Example 3
    # price3 = predict_car_price(2015, "Lexus", "Rx 350",  "Automatic", "Local Used")
    # print("\n 2015 Lexus RX 350 (Local used Automatic)")
    # print(f"  ₦{float(price3):,.2f}")

    # Example 1
    price1 = predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    low, high = price1
    print("\n 2015 Toyota Camry (Local used Automatic)")
    print(f"  ₦{low:,.2f} - ₦{high:,.2f}")

    # Example 2
    price2 = predict_car_price(2010, "Honda", "Accord", "Automatic", "Local Used")
    low, high = price2
    print("\n 2010 Honda Accord (Local used Automatic)")
    print(f"  ₦{low:,.2f} - ₦{high:,.2f}")

    # Example 3
    price3 = predict_car_price(2015, "Lexus", "Rx 350", "Automatic", "Local Used")
    low, high = price3
    print("\n 2015 Lexus RX 350 (Local used Automatic)")
    print(f"  ₦{low:,.2f} - ₦{high:,.2f}")
    print("=" * 60)

In [429]:
def test_rm_model_prediction():
    print("\n" + "=" * 60)
    print("TESTING PREDICTION")
    print("=" * 60)

    #Example 1
    price1 = rm_model_predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    print("\n 2015 Toyota Camry( Local used Automatic)")
    print(f"  ₦{float(price1):,.2f}")
        
    #Example 2
    price2 = rm_model_predict_car_price(2010, "Honda", "Accord", "Automatic", "Local Used")
    print("\n 2010 Honda Accord ( Local used Automatic)")
    print(f"  ₦{float(price2):,.2f}")

    #Example 3
    price3 = rm_model_predict_car_price(2015, "Lexus", "Rx 350",  "Automatic", "Local Used")
    print("\n 2015 Lexus RX 350 (Local used Automatic)")
    print(f"  ₦{float(price3):,.2f}")
    
    print("=" * 60)

In [430]:
def main():
    file_path = "jiji_car_dataset.csv"

    df = load_and_explore_dataset(file_path)

    df_preprocessed, label_encoder = preprocessing_data(df)

    X, y, feature_columns = feature_data(df_preprocessed)

    X_train, X_test, y_train, y_test = split_data(X, y)

    X_train_scaled, X_test_scaled, scaler = scale_feature(X_train, X_test)

    model = train_model(X_train_scaled, y_train, feature_columns)

    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test )

    save_model_artifact(model,scaler, label_encoder, feature_columns) 

    test_prediction()

    rm_model = train_random_forest_model(X_train_scaled, y_train, feature_columns)

    rm_metrics = evaluate_rm_model(rm_model, X_train_scaled,X_test_scaled, y_train, y_test) 
    
    save_rm_model_artifact(rm_model) 

    test_rm_model_prediction()

In [431]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATASET
Shape of the dataset:
(1814, 7)

check for missing value
title           0
make            0
model           0
year            0
condition       0
transmission    0
price           0
dtype: int64
\First five five row
                                           title           make  \
0  Mercedes-Benz GLK-Class 350 4MATIC 2011 Black  Mercedes-Benz   
1                 Dodge Charger SE AWD 2016 Gray          Dodge   
2                        Toyota Camry 2010 Black         Toyota   
3                    Toyota Venza AWD 2010 Brown         Toyota   
4                         Honda Civic 2007 Black          Honda   

           model    year     condition transmission       price  
0  Glk-Class 350  2011.0    Local Used    Automatic   9500000.0  
1     Charger Se  2016.0  Foreign Used    Automatic  17000000.0  
2          Camry  2010.0    Local Used    Automatic   9492187.0  
3      Venza Awd  2010.0    Local Used    Automatic   8700000.0  
4          Civic  2007.0 